# 05 - Feature Engineering

In [22]:
import pandas as pd
import geopandas as gpd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [23]:
# Load cleaned data
df = pd.read_csv("cleaned.csv", dtype={"CloseMonth": str})
months = sorted(df["CloseMonth"].unique())
old_features = [c for c in df.columns if c not in ["ClosePrice", "CloseMonth"]]
df.shape

(60477, 74)

## New features

In [24]:
# Bedrooms per bathroom (homes listed with 0 baths get 0)
df["BedBathRatio"] = (df["BedroomsTotal"] / df["BathroomsTotalInteger"]).replace(float("inf"), 0).fillna(0)

# Age of the home in years at the time of sale
df["Age"] = df["CloseMonth"].str[:4].astype(int) - df["YearBuilt"]

df[["BedBathRatio", "Age"]].describe()

,BedBathRatio,Age
count,60477.000000,60477.000000
mean,1.452402,49.611522
std,0.473458,27.729547
min,0.000000,-1.000000
25%,1.000000,27.000000
50%,1.500000,49.000000
75%,1.500000,70.000000
max,6.000000,225.000000


## School districts

In [25]:
districts = gpd.read_file("school_districts/DistrictAreas2425.shp")
points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]), index=df.index, crs="EPSG:4326")
points = points.to_crs(districts.crs)

joined = gpd.sjoin(points, districts[["DistrictNa", "DistrictTy", "geometry"]], how="left", predicate="within")
joined["DistrictTy"].value_counts(dropna=False)

DistrictTy
Unified       45975
Elementary    14759
High          14430
NaN              27
Name: count, dtype: int64

In [26]:
# Homes in both an elementary and a high school district: keep the unified district if there is one, otherwise the elementary one
joined["priority"] = joined["DistrictTy"].map({"Unified": 0, "Elementary": 1, "High": 2})
joined = joined.sort_values("priority").groupby(level=0).first()

df["SchoolDistrict"] = joined["DistrictNa"].fillna("None")
df[["Latitude", "Longitude", "SchoolDistrict"]].head(5)

,Latitude,Longitude,SchoolDistrict
0,37.871927,-122.029871,San Ramon Valley Unified
1,34.150680,-118.580650,Los Angeles Unified
2,37.306405,-121.835428,Franklin-McKinley Elementary
3,37.272636,-121.921351,Cambrian
4,37.303942,-121.935424,Campbell Union


In [27]:
df = pd.get_dummies(df, columns=["SchoolDistrict"], drop_first=True, dtype=int)
district_features = [c for c in df.columns if c.startswith("SchoolDistrict_")]
df.shape

(60477, 659)

## Train/test split

In [28]:
# Test on the most recent month, train on the X months before it
X = 4
train = df[df["CloseMonth"].isin(months[-1 - X:-1])].copy()
test = df[df["CloseMonth"] == months[-1]].copy()
train.shape, test.shape

((38411, 659), (11788, 659))

In [29]:
# Normalize numeric features using the training set's mean and std
num_cols = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt",
            "GarageSpaces", "ParkingTotal", "AssociationFee", "Latitude", "Longitude", "BedBathRatio", "Age"]
mean, std = train[num_cols].mean(), train[num_cols].std()
train[num_cols] = (train[num_cols] - mean) / std
test[num_cols] = (test[num_cols] - mean) / std

## Models

In [30]:
feature_sets = {
    "old": old_features,
    "old + bed/bath ratio + age": old_features + ["BedBathRatio", "Age"],
    "old + bed/bath ratio + age + school district": old_features + ["BedBathRatio", "Age"] + district_features,
}

models = {
    "LinearRegression": LinearRegression,
    "DecisionTree": DecisionTreeRegressor,
    "RandomForest": lambda: RandomForestRegressor(n_jobs=-1),
}

rows = []
for set_name, features in feature_sets.items():
    for model_name, make_model in models.items():
        model = make_model()
        model.fit(train[features], train["ClosePrice"])
        test_r2 = r2_score(test["ClosePrice"], model.predict(test[features]))
        rows.append((set_name, model_name, test_r2))

## Results

In [ ]:
for set_name in feature_sets:
    print(f"Features: {set_name}")
    for s, model_name, test_r2 in rows:
        if s == set_name:
            print(f"{model_name} test R2: {test_r2:.3f}")
    print()

Features: old
  LinearRegression test R²: 0.595
  DecisionTree test R²: 0.650
  RandomForest test R²: 0.824

Features: old + bed/bath ratio + age
  LinearRegression test R²: 0.603
  DecisionTree test R²: 0.647
  RandomForest test R²: 0.824

Features: old + bed/bath ratio + age + school district
  LinearRegression test R²: 0.732
  DecisionTree test R²: 0.652
  RandomForest test R²: 0.831

